# 🔧 Maintenance et Monitoring - AskMe Search

Ce notebook permet de maintenir et surveiller le système OpenSearch en production.

**Use cases :**
- 📊 Monitoring des performances et de la santé
- 🔍 Diagnostic et résolution de problèmes
- 🧹 Maintenance et nettoyage
- 💾 Sauvegarde et restauration
- 📈 Analyse d'utilisation
- ⚠️ Alertes et seuils

## 📦 Configuration

In [ ]:
import sys
sys.path.append('../scripts')

from simple_indexer import SimpleIndexer
from pathlib import Path
import requests
import json
from typing import List, Dict, Optional
from datetime import datetime
import time

# Configuration
OPENSEARCH_URL = "http://localhost:9200"
CLIENTS_DATA_DIR = "../clients-data"

print("🔧 Notebook Maintenance et Monitoring chargé")
print(f"   OpenSearch: {OPENSEARCH_URL}")
print(f"   Architecture: Multi-clients avec documents UUID + PJ multiples")

## 🏥 Contrôle de Santé du Système

In [ ]:
def comprehensive_health_check() -> Dict:
    """Contrôle de santé complet du système OpenSearch"""
    
    print("🏥 Contrôle de Santé Complet du Système")
    print("=" * 60)
    
    health_report = {
        'timestamp': datetime.now().isoformat(),
        'overall_status': 'unknown',
        'cluster': {},
        'nodes': {},
        'indices': {},
        'performance': {},
        'storage': {},
        'alerts': []
    }
    
    # 1. Santé du cluster
    print("\n🔗 1. Santé du Cluster:")
    try:
        response = requests.get(f"{OPENSEARCH_URL}/_cluster/health")
        if response.status_code == 200:
            cluster_health = response.json()
            health_report['cluster'] = cluster_health
            
            status = cluster_health.get('status', 'unknown')
            if status == 'green':
                print(f"   ✅ Status: {status} (Excellent)")
            elif status == 'yellow':
                print(f"   ⚠️ Status: {status} (Attention requise)")
                health_report['alerts'].append("Cluster en status yellow")
            else:
                print(f"   ❌ Status: {status} (Problème critique)")
                health_report['alerts'].append(f"Cluster en status {status}")
            
            print(f"   📊 Nœuds: {cluster_health.get('number_of_nodes', 0)}")
            print(f"   📊 Nœuds données: {cluster_health.get('number_of_data_nodes', 0)}")
            print(f"   📊 Shards actifs: {cluster_health.get('active_shards', 0)}")
            print(f"   📊 Index: {cluster_health.get('number_of_indices', 0)}")
            
            # Alertes spécifiques
            if cluster_health.get('relocating_shards', 0) > 0:
                health_report['alerts'].append(f"{cluster_health['relocating_shards']} shards en relocation")
            
            if cluster_health.get('unassigned_shards', 0) > 0:
                health_report['alerts'].append(f"{cluster_health['unassigned_shards']} shards non assignés")
        else:
            print(f"   ❌ Impossible de récupérer la santé du cluster: {response.status_code}")
            health_report['alerts'].append("Cluster inaccessible")
    except Exception as e:
        print(f"   ❌ Erreur cluster: {e}")
        health_report['alerts'].append(f"Erreur cluster: {e}")
    
    # 2. Informations sur les nœuds
    print("\n🖥️ 2. Informations Nœuds:")
    try:
        response = requests.get(f"{OPENSEARCH_URL}/_nodes/stats")
        if response.status_code == 200:
            nodes_stats = response.json()
            health_report['nodes'] = nodes_stats
            
            nodes = nodes_stats.get('nodes', {})
            print(f"   📊 Nombre de nœuds: {len(nodes)}")
            
            for node_id, node_info in nodes.items():
                name = node_info.get('name', 'Unknown')
                
                # Mémoire
                jvm = node_info.get('jvm', {}).get('mem', {})
                heap_used_percent = jvm.get('heap_used_percent', 0)
                
                # CPU
                os_info = node_info.get('os', {}).get('cpu', {})
                cpu_percent = os_info.get('percent', 0)
                
                print(f"   🖥️ {name}:")
                print(f"      🧠 Heap JVM: {heap_used_percent}%")
                print(f"      ⚡ CPU: {cpu_percent}%")
                
                # Alertes performance
                if heap_used_percent > 85:
                    health_report['alerts'].append(f"Nœud {name}: Heap JVM élevé ({heap_used_percent}%)")
                
                if cpu_percent > 80:
                    health_report['alerts'].append(f"Nœud {name}: CPU élevé ({cpu_percent}%)")
    except Exception as e:
        print(f"   ❌ Erreur nœuds: {e}")
        health_report['alerts'].append(f"Erreur nœuds: {e}")
    
    # 3. État des index
    print("\n📚 3. État des Index:")
    try:
        indexer = SimpleIndexer()
        all_indexes = indexer.list_indexes()
        
        health_report['indices']['count'] = len(all_indexes)
        health_report['indices']['list'] = all_indexes
        
        print(f"   📊 Nombre d'index: {len(all_indexes)}")
        
        total_docs = 0
        total_size_mb = 0
        
        for index_name in all_indexes:
            if index_name.startswith('askme-'):
                index_indexer = SimpleIndexer(index_name=index_name)
                stats = index_indexer.get_index_stats()
                
                if stats:
                    docs = stats.get('documents_count', 0)
                    size_mb = stats.get('size_mb', 0)
                    
                    total_docs += docs
                    total_size_mb += size_mb
                    
                    print(f"   📄 {index_name}: {docs} docs, {size_mb} MB")
                    
                    # Alertes index
                    if docs == 0:
                        health_report['alerts'].append(f"Index {index_name} vide")
                    elif size_mb > 1000:  # > 1GB
                        health_report['alerts'].append(f"Index {index_name} volumineux ({size_mb} MB)")
        
        health_report['indices']['total_documents'] = total_docs
        health_report['indices']['total_size_mb'] = total_size_mb
        
        print(f"   📊 Total: {total_docs} documents, {total_size_mb:.1f} MB")
        
    except Exception as e:
        print(f"   ❌ Erreur index: {e}")
        health_report['alerts'].append(f"Erreur index: {e}")
    
    # 4. Test de performance basique
    print("\n⚡ 4. Test de Performance:")
    try:
        test_indexer = SimpleIndexer()
        
        # Test de recherche simple
        start_time = time.time()
        test_results = test_indexer.search("test", size=1)
        search_time = time.time() - start_time
        
        health_report['performance']['search_time'] = search_time
        
        print(f"   🔍 Temps recherche test: {search_time:.3f}s")
        
        if search_time > 1.0:
            health_report['alerts'].append(f"Recherche lente ({search_time:.3f}s)")
        
    except Exception as e:
        print(f"   ❌ Erreur test performance: {e}")
        health_report['alerts'].append(f"Erreur test performance: {e}")
    
    # Statut global
    if len(health_report['alerts']) == 0:
        health_report['overall_status'] = 'healthy'
        status_icon = "✅"
        status_text = "SYSTÈME SAIN"
    elif len(health_report['alerts']) <= 2:
        health_report['overall_status'] = 'warning'
        status_icon = "⚠️"
        status_text = "ATTENTION REQUISE"
    else:
        health_report['overall_status'] = 'critical'
        status_icon = "❌"
        status_text = "PROBLÈMES CRITIQUES"
    
    print(f"\n{status_icon} STATUT GLOBAL: {status_text}")
    print("=" * 60)
    
    if health_report['alerts']:
        print(f"⚠️ Alertes ({len(health_report['alerts'])}):")
        for alert in health_report['alerts']:
            print(f"   • {alert}")
    else:
        print(f"🎉 Aucune alerte - Système en parfait état !")
    
    return health_report

# Exécuter le contrôle de santé
health_status = comprehensive_health_check()

## 📊 Monitoring Continu

In [ ]:
def monitor_system_metrics(duration_minutes: int = 5, interval_seconds: int = 30) -> List[Dict]:
    """Surveiller les métriques système en continu"""
    
    print(f"📊 Monitoring Continu")
    print(f"⏱️ Durée: {duration_minutes} minutes")
    print(f"🔄 Intervalle: {interval_seconds} secondes")
    print(f"📈 {int(duration_minutes * 60 / interval_seconds)} mesures prévues")
    print("=" * 60)
    
    metrics_history = []
    start_time = time.time()
    end_time = start_time + (duration_minutes * 60)
    
    measurement_count = 0
    
    try:
        while time.time() < end_time:
            measurement_count += 1
            current_time = datetime.now()
            
            print(f"\n📊 Mesure #{measurement_count} - {current_time.strftime('%H:%M:%S')}")
            
            metrics = {
                'timestamp': current_time.isoformat(),
                'measurement': measurement_count
            }
            
            # Santé du cluster
            try:
                response = requests.get(f"{OPENSEARCH_URL}/_cluster/health")
                if response.status_code == 200:
                    health = response.json()
                    metrics['cluster_status'] = health.get('status', 'unknown')
                    metrics['active_shards'] = health.get('active_shards', 0)
                    print(f"   🔗 Cluster: {metrics['cluster_status']} ({metrics['active_shards']} shards)")
                else:
                    metrics['cluster_status'] = 'error'
                    print(f"   ❌ Cluster inaccessible")
            except:
                metrics['cluster_status'] = 'error'
                print(f"   ❌ Erreur cluster")
            
            # Performance de recherche
            try:
                indexer = SimpleIndexer()
                search_start = time.time()
                indexer.search("test", size=1)
                search_time = time.time() - search_start
                
                metrics['search_time'] = search_time
                print(f"   🔍 Recherche: {search_time:.3f}s")
                
                if search_time > 0.5:
                    print(f"   ⚠️ Recherche lente détectée")
            except:
                metrics['search_time'] = None
                print(f"   ❌ Erreur test recherche")
            
            # Stats nœuds (simplifié)
            try:
                response = requests.get(f"{OPENSEARCH_URL}/_nodes/stats")
                if response.status_code == 200:
                    nodes_stats = response.json()
                    nodes = nodes_stats.get('nodes', {})
                    
                    if nodes:
                        # Prendre le premier nœud
                        first_node = list(nodes.values())[0]
                        
                        jvm = first_node.get('jvm', {}).get('mem', {})
                        heap_percent = jvm.get('heap_used_percent', 0)
                        
                        os_info = first_node.get('os', {}).get('cpu', {})
                        cpu_percent = os_info.get('percent', 0)
                        
                        metrics['heap_percent'] = heap_percent
                        metrics['cpu_percent'] = cpu_percent
                        
                        print(f"   🧠 Heap: {heap_percent}% | ⚡ CPU: {cpu_percent}%")
                        
                        if heap_percent > 80:
                            print(f"   ⚠️ Heap élevé détecté")
                        if cpu_percent > 80:
                            print(f"   ⚠️ CPU élevé détecté")
            except:
                print(f"   ❌ Erreur stats nœuds")
            
            metrics_history.append(metrics)
            
            # Attendre l'intervalle suivant
            if time.time() < end_time:
                time.sleep(interval_seconds)
    
    except KeyboardInterrupt:
        print(f"\n⏹️ Monitoring interrompu par l'utilisateur")
    
    # Analyse des résultats
    print(f"\n📈 ANALYSE DU MONITORING:")
    print("=" * 50)
    print(f"📊 Mesures collectées: {len(metrics_history)}")
    
    if metrics_history:
        # Analyse des temps de recherche
        search_times = [m['search_time'] for m in metrics_history if m.get('search_time') is not None]
        if search_times:
            avg_search = sum(search_times) / len(search_times)
            max_search = max(search_times)
            min_search = min(search_times)
            print(f"🔍 Recherche - Moy: {avg_search:.3f}s | Max: {max_search:.3f}s | Min: {min_search:.3f}s")
        
        # Analyse heap
        heap_values = [m['heap_percent'] for m in metrics_history if m.get('heap_percent') is not None]
        if heap_values:
            avg_heap = sum(heap_values) / len(heap_values)
            max_heap = max(heap_values)
            print(f"🧠 Heap JVM - Moy: {avg_heap:.1f}% | Max: {max_heap:.1f}%")
        
        # Statut cluster
        cluster_statuses = [m['cluster_status'] for m in metrics_history]
        status_counts = {}
        for status in cluster_statuses:
            status_counts[status] = status_counts.get(status, 0) + 1
        
        print(f"🔗 Statuts cluster: {dict(status_counts)}")
    
    return metrics_history

def quick_system_check() -> Dict:
    """Contrôle rapide du système (version légère)"""
    
    print("⚡ Contrôle Rapide du Système")
    print("=" * 40)
    
    check = {
        'timestamp': datetime.now().isoformat(),
        'cluster_ok': False,
        'search_ok': False,
        'performance_ok': False,
        'overall_ok': False
    }
    
    # Test cluster
    try:
        response = requests.get(f"{OPENSEARCH_URL}/_cluster/health", timeout=5)
        if response.status_code == 200:
            health = response.json()
            status = health.get('status', 'red')
            check['cluster_ok'] = status in ['green', 'yellow']
            print(f"🔗 Cluster: {'✅' if check['cluster_ok'] else '❌'} ({status})")
        else:
            print(f"🔗 Cluster: ❌ (HTTP {response.status_code})")
    except:
        print(f"🔗 Cluster: ❌ (Connexion impossible)")
    
    # Test recherche
    try:
        indexer = SimpleIndexer()
        start_time = time.time()
        result = indexer.search("test", size=1)
        search_time = time.time() - start_time
        
        check['search_ok'] = search_time < 2.0  # Moins de 2 secondes
        check['search_time'] = search_time
        print(f"🔍 Recherche: {'✅' if check['search_ok'] else '❌'} ({search_time:.3f}s)")
        
        check['performance_ok'] = search_time < 0.5  # Performance optimale
        print(f"⚡ Performance: {'✅' if check['performance_ok'] else '⚠️'} ({'Optimale' if check['performance_ok'] else 'Acceptable'})")
    except Exception as e:
        print(f"🔍 Recherche: ❌ (Erreur: {e})")
    
    # Statut global
    check['overall_ok'] = check['cluster_ok'] and check['search_ok']
    
    print(f"\n{'✅' if check['overall_ok'] else '❌'} Système: {'Opérationnel' if check['overall_ok'] else 'Problèmes détectés'}")
    
    return check

# Contrôle rapide
print("📊 Tests de monitoring:")
print("=" * 50)

quick_status = quick_system_check()

print(f"\n💡 Pour monitoring continu (ATTENTION: peut être long):")
print(f"   monitor_system_metrics(duration_minutes=2, interval_seconds=15)")
print(f"\n⚠️ Monitoring continu désactivé - Décommentez pour activer:")

# DÉCOMMENTEZ POUR MONITORING CONTINU (peut être long):
# print("\n" + "="*60)
# monitoring_data = monitor_system_metrics(duration_minutes=1, interval_seconds=10)

## 🧹 Maintenance et Nettoyage

In [ ]:
def analyze_storage_usage() -> Dict:
    """Analyser l'utilisation de l'espace de stockage"""
    
    print("💾 Analyse de l'Utilisation du Stockage")
    print("=" * 60)
    
    storage_analysis = {
        'timestamp': datetime.now().isoformat(),
        'total_size_mb': 0,
        'total_documents': 0,
        'indices': [],
        'recommendations': []
    }
    
    try:
        indexer = SimpleIndexer()
        all_indexes = indexer.list_indexes()
        
        print(f"📊 Analyse de {len(all_indexes)} index:")
        print()
        
        for index_name in all_indexes:
            if index_name.startswith('askme-') or index_name == 'askme-documents':
                index_indexer = SimpleIndexer(index_name=index_name)
                stats = index_indexer.get_index_stats()
                
                if stats:
                    docs = stats.get('documents_count', 0)
                    size_mb = stats.get('size_mb', 0)
                    size_bytes = stats.get('size_bytes', 0)
                    
                    storage_analysis['total_size_mb'] += size_mb
                    storage_analysis['total_documents'] += docs
                    
                    # Calcul de la taille moyenne par document
                    avg_size_per_doc = size_bytes / docs if docs > 0 else 0
                    
                    index_info = {
                        'name': index_name,
                        'documents': docs,
                        'size_mb': size_mb,
                        'avg_size_per_doc': avg_size_per_doc
                    }
                    storage_analysis['indices'].append(index_info)
                    
                    # Affichage
                    if size_mb > 0:
                        print(f"📚 {index_name}:")
                        print(f"   📄 Documents: {docs:,}")
                        print(f"   💾 Taille: {size_mb:.2f} MB")
                        print(f"   📊 Moy/doc: {avg_size_per_doc/1024:.1f} KB")
                        
                        # Recommendations
                        if docs == 0:
                            storage_analysis['recommendations'].append(f"Index {index_name} est vide - considérer la suppression")
                        elif size_mb > 500:  # > 500MB
                            storage_analysis['recommendations'].append(f"Index {index_name} très volumineux ({size_mb:.1f} MB) - vérifier le contenu")
                        elif avg_size_per_doc > 100000:  # > 100KB par doc
                            storage_analysis['recommendations'].append(f"Index {index_name} a des documents volumineux (moy: {avg_size_per_doc/1024:.1f} KB)")
                        
                        print()
                    else:
                        print(f"📚 {index_name}: Vide ou erreur")
        
        # Résumé global
        print(f"📊 RÉSUMÉ GLOBAL:")
        print("=" * 40)
        print(f"💾 Espace total utilisé: {storage_analysis['total_size_mb']:.2f} MB")
        print(f"📄 Total documents: {storage_analysis['total_documents']:,}")
        
        if storage_analysis['total_documents'] > 0:
            avg_global = (storage_analysis['total_size_mb'] * 1024 * 1024) / storage_analysis['total_documents']
            print(f"📊 Taille moyenne globale: {avg_global/1024:.1f} KB/document")
        
        # Recommandations
        if storage_analysis['recommendations']:
            print(f"\n💡 RECOMMANDATIONS ({len(storage_analysis['recommendations'])}):")
            for i, rec in enumerate(storage_analysis['recommendations'], 1):
                print(f"   {i}. {rec}")
        else:
            print(f"\n✅ Aucune recommandation - Utilisation optimale")
        
    except Exception as e:
        print(f"❌ Erreur analyse stockage: {e}")
        storage_analysis['error'] = str(e)
    
    return storage_analysis

def cleanup_empty_indices() -> Dict:
    """Nettoyer les index vides (avec confirmation)"""
    
    print("🧹 Nettoyage des Index Vides")
    print("=" * 50)
    
    cleanup_report = {
        'empty_indices_found': [],
        'deleted_indices': [],
        'errors': []
    }
    
    try:
        indexer = SimpleIndexer()
        all_indexes = indexer.list_indexes()
        
        # Identifier les index vides
        for index_name in all_indexes:
            if index_name.startswith('askme-') or index_name == 'askme-documents':
                index_indexer = SimpleIndexer(index_name=index_name)
                stats = index_indexer.get_index_stats()
                
                if stats and stats.get('documents_count', 0) == 0:
                    cleanup_report['empty_indices_found'].append(index_name)
        
        if cleanup_report['empty_indices_found']:
            print(f"📋 Index vides trouvés ({len(cleanup_report['empty_indices_found'])}):")
            for idx in cleanup_report['empty_indices_found']:
                print(f"   📁 {idx}")
            
            print(f"\n⚠️ ATTENTION: Suppression désactivée pour sécurité")
            print(f"⚠️ Pour supprimer, décommentez le code ci-dessous:")
            print()
            
            # CODE DE SUPPRESSION (commenté pour sécurité)
            # print(f"🗑️ Suppression des index vides...")
            # for index_name in cleanup_report['empty_indices_found']:
            #     try:
            #         response = requests.delete(f"{OPENSEARCH_URL}/{index_name}")
            #         if response.status_code == 200:
            #             cleanup_report['deleted_indices'].append(index_name)
            #             print(f"   ✅ Supprimé: {index_name}")
            #         else:
            #             error_msg = f"Erreur suppression {index_name}: HTTP {response.status_code}"
            #             cleanup_report['errors'].append(error_msg)
            #             print(f"   ❌ {error_msg}")
            #     except Exception as e:
            #         error_msg = f"Erreur suppression {index_name}: {e}"
            #         cleanup_report['errors'].append(error_msg)
            #         print(f"   ❌ {error_msg}")
            
            print(f"💡 Pour supprimer manuellement:")
            for idx in cleanup_report['empty_indices_found']:
                print(f"   curl -X DELETE \"{OPENSEARCH_URL}/{idx}\"")
        else:
            print(f"✅ Aucun index vide trouvé")
    
    except Exception as e:
        error_msg = f"Erreur nettoyage: {e}"
        cleanup_report['errors'].append(error_msg)
        print(f"❌ {error_msg}")
    
    return cleanup_report

def force_refresh_indices() -> Dict:
    """Forcer le rafraîchissement de tous les index"""
    
    print("🔄 Rafraîchissement Forcé des Index")
    print("=" * 50)
    
    refresh_report = {
        'refreshed_indices': [],
        'errors': []
    }
    
    try:
        indexer = SimpleIndexer()
        all_indexes = indexer.list_indexes()
        
        for index_name in all_indexes:
            if index_name.startswith('askme-') or index_name == 'askme-documents':
                try:
                    response = requests.post(f"{OPENSEARCH_URL}/{index_name}/_refresh")
                    if response.status_code == 200:
                        refresh_report['refreshed_indices'].append(index_name)
                        print(f"   ✅ Rafraîchi: {index_name}")
                    else:
                        error_msg = f"Erreur refresh {index_name}: HTTP {response.status_code}"
                        refresh_report['errors'].append(error_msg)
                        print(f"   ❌ {error_msg}")
                except Exception as e:
                    error_msg = f"Erreur refresh {index_name}: {e}"
                    refresh_report['errors'].append(error_msg)
                    print(f"   ❌ {error_msg}")
        
        print(f"\n📊 Résumé:")
        print(f"   ✅ Rafraîchis: {len(refresh_report['refreshed_indices'])}")
        print(f"   ❌ Erreurs: {len(refresh_report['errors'])}")
    
    except Exception as e:
        error_msg = f"Erreur générale refresh: {e}"
        refresh_report['errors'].append(error_msg)
        print(f"❌ {error_msg}")
    
    return refresh_report

# Exécuter l'analyse de stockage
storage_report = analyze_storage_usage()

print(f"\n" + "="*60)

# Nettoyage (sécurisé)
cleanup_report = cleanup_empty_indices()

print(f"\n" + "="*60)

# Rafraîchissement
refresh_report = force_refresh_indices()

## 💾 Sauvegarde et Restauration

In [ ]:
def export_index_metadata(index_name: str) -> Dict:
    """Exporter les métadonnées d'un index pour sauvegarde"""
    
    print(f"💾 Export des métadonnées: {index_name}")
    print("=" * 50)
    
    metadata = {
        'index_name': index_name,
        'export_timestamp': datetime.now().isoformat(),
        'mapping': None,
        'settings': None,
        'stats': None,
        'document_sample': [],
        'error': None
    }
    
    try:
        # Mapping de l'index
        response = requests.get(f"{OPENSEARCH_URL}/{index_name}/_mapping")
        if response.status_code == 200:
            metadata['mapping'] = response.json()
            print(f"   ✅ Mapping exporté")
        else:
            print(f"   ⚠️ Impossible d'exporter le mapping: HTTP {response.status_code}")
        
        # Settings de l'index
        response = requests.get(f"{OPENSEARCH_URL}/{index_name}/_settings")
        if response.status_code == 200:
            metadata['settings'] = response.json()
            print(f"   ✅ Settings exportés")
        else:
            print(f"   ⚠️ Impossible d'exporter les settings: HTTP {response.status_code}")
        
        # Statistiques
        indexer = SimpleIndexer(index_name=index_name)
        stats = indexer.get_index_stats()
        if stats:
            metadata['stats'] = stats
            print(f"   ✅ Statistiques exportées")
            print(f"      📄 {stats.get('documents_count', 0)} documents")
            print(f"      💾 {stats.get('size_mb', 0)} MB")
        
        # Échantillon de documents (pour validation)
        docs_response = indexer.list_documents(size=5)
        if docs_response and 'hits' in docs_response:
            hits = docs_response['hits']['hits']
            for hit in hits:
                doc_sample = {
                    'id': hit['_id'],
                    'source': hit['_source']
                }
                metadata['document_sample'].append(doc_sample)
            
            print(f"   ✅ Échantillon de {len(hits)} documents")
        
        print(f"\n✅ Export terminé avec succès")
    
    except Exception as e:
        error_msg = f"Erreur export {index_name}: {e}"
        metadata['error'] = error_msg
        print(f"❌ {error_msg}")
    
    return metadata

def create_backup_script(backup_dir: str = "./backups") -> str:
    """Créer un script de sauvegarde automatisé"""
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    script_content = f'''#!/bin/bash
# Script de sauvegarde AskMe Search - Généré le {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

BACKUP_DIR="{backup_dir}"
TIMESTAMP="{timestamp}"
OPENSEARCH_URL="{OPENSEARCH_URL}"

echo "🔄 Début de la sauvegarde AskMe Search - $TIMESTAMP"

# Créer le répertoire de sauvegarde
mkdir -p "$BACKUP_DIR/$TIMESTAMP"

# Sauvegarder la santé du cluster
echo "📊 Sauvegarde santé cluster..."
curl -s "$OPENSEARCH_URL/_cluster/health" > "$BACKUP_DIR/$TIMESTAMP/cluster_health.json"

# Sauvegarder la liste des index
echo "📋 Sauvegarde liste des index..."
curl -s "$OPENSEARCH_URL/_cat/indices?format=json" > "$BACKUP_DIR/$TIMESTAMP/indices_list.json"

# Sauvegarder les mappings et settings de chaque index askme-*
echo "💾 Sauvegarde des métadonnées des index..."
'''
    
    # Ajouter les commandes pour chaque index
    try:
        indexer = SimpleIndexer()
        all_indexes = indexer.list_indexes()
        
        for index_name in all_indexes:
            if index_name.startswith('askme-') or index_name == 'askme-documents':
                script_content += f'''
echo "  📚 Sauvegarde {index_name}..."
curl -s "$OPENSEARCH_URL/{index_name}/_mapping" > "$BACKUP_DIR/$TIMESTAMP/{index_name}_mapping.json"
curl -s "$OPENSEARCH_URL/{index_name}/_settings" > "$BACKUP_DIR/$TIMESTAMP/{index_name}_settings.json"
curl -s "$OPENSEARCH_URL/{index_name}/_stats" > "$BACKUP_DIR/$TIMESTAMP/{index_name}_stats.json"
'''
    except:
        script_content += "\n# Erreur: Impossible de lister les index\n"
    
    script_content += f'''
# Créer un résumé
echo "📋 Création du résumé..."
cat > "$BACKUP_DIR/$TIMESTAMP/backup_info.txt" << EOF
Sauvegarde AskMe Search
======================
Date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
Timestamp: $TIMESTAMP
OpenSearch URL: $OPENSEARCH_URL

Contenu de la sauvegarde:
- cluster_health.json: Santé du cluster
- indices_list.json: Liste des index
- *_mapping.json: Mappings des index
- *_settings.json: Settings des index
- *_stats.json: Statistiques des index

Pour restaurer un index:
1. Créer l'index avec les settings et mappings
2. Réindexer les documents depuis les sources

EOF

echo "✅ Sauvegarde terminée: $BACKUP_DIR/$TIMESTAMP"
echo "📊 Taille de la sauvegarde:"
du -sh "$BACKUP_DIR/$TIMESTAMP"
'''
    
    return script_content

def disaster_recovery_plan() -> str:
    """Générer un plan de récupération d'urgence"""
    
    plan = f'''
# 🚨 Plan de Récupération d'Urgence - AskMe Search
Généré le: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

## 🔍 Diagnostic Rapide

### 1. Vérifier l'état du cluster
```bash
curl "{OPENSEARCH_URL}/_cluster/health?pretty"
```

### 2. Lister les index existants
```bash
curl "{OPENSEARCH_URL}/_cat/indices?v"
```

### 3. Vérifier l'espace disque
```bash
df -h
docker system df
```

## 🔧 Actions de Récupération

### Problème: Cluster Rouge
1. Redémarrer OpenSearch: `docker-compose restart`
2. Vérifier les logs: `docker-compose logs opensearch`
3. Libérer l'espace disque si nécessaire

### Problème: Index Corrompus
1. Identifier les index problématiques
2. Supprimer et recréer l'index
3. Réindexer depuis les sources avec les notebooks

### Problème: Performances Dégradées
1. Forcer le refresh: `curl -X POST "{OPENSEARCH_URL}/_refresh"`
2. Optimiser les index: `curl -X POST "{OPENSEARCH_URL}/_forcemerge"`
3. Redémarrer le conteneur

### Problème: Données Perdues
1. Vérifier les sauvegardes disponibles
2. Recréer les index avec les métadonnées sauvegardées
3. Réindexer tous les documents depuis les sources

## 📞 Contacts d'Urgence
- Administrateur système: [À REMPLIR]
- Support technique: [À REMPLIR]

## 📋 Checklist Post-Incident
- [ ] Système restauré et fonctionnel
- [ ] Tests de recherche OK
- [ ] Tous les index présents
- [ ] Performances normales
- [ ] Sauvegarde post-restauration
- [ ] Documentation de l'incident
'''
    
    return plan

# Démonstration des outils de sauvegarde
print("💾 Outils de Sauvegarde et Récupération:")
print("=" * 60)

# Export d'un index exemple
try:
    indexer = SimpleIndexer()
    all_indexes = indexer.list_indexes()
    askme_indexes = [idx for idx in all_indexes if idx.startswith('askme-') or idx == 'askme-documents']
    
    if askme_indexes:
        sample_index = askme_indexes[0]
        print(f"📄 Export de métadonnées pour: {sample_index}")
        metadata = export_index_metadata(sample_index)
        
        print(f"\n💾 Métadonnées collectées:")
        print(f"   📊 Mapping: {'✅' if metadata['mapping'] else '❌'}")
        print(f"   ⚙️ Settings: {'✅' if metadata['settings'] else '❌'}")
        print(f"   📈 Stats: {'✅' if metadata['stats'] else '❌'}")
        print(f"   📄 Échantillons: {len(metadata.get('document_sample', []))}")
    else:
        print(f"⚠️ Aucun index AskMe trouvé pour l'export")
except Exception as e:
    print(f"❌ Erreur export: {e}")

print(f"\n🔧 Génération du script de sauvegarde:")
backup_script = create_backup_script()
print(f"✅ Script généré ({len(backup_script)} caractères)")
print(f"💡 Pour utiliser: copiez le script dans un fichier .sh et exécutez-le")

print(f"\n🚨 Plan de récupération d'urgence:")
recovery_plan = disaster_recovery_plan()
print(f"✅ Plan généré ({len(recovery_plan)} caractères)")
print(f"💡 Conservez ce plan dans vos procédures d'urgence")

## 🔍 Diagnostic et Résolution de Problèmes

In [ ]:
def diagnose_search_issues(test_queries: List[str] = None) -> Dict:
    """Diagnostiquer les problèmes de recherche"""
    
    if test_queries is None:
        test_queries = ["test", "configuration", "document", "système"]
    
    print("🔍 Diagnostic des Problèmes de Recherche")
    print("=" * 60)
    
    diagnosis = {
        'timestamp': datetime.now().isoformat(),
        'test_queries': test_queries,
        'search_results': [],
        'performance_issues': [],
        'content_issues': [],
        'recommendations': []
    }
    
    try:
        indexer = SimpleIndexer()
        
        print(f"🧪 Test de {len(test_queries)} requêtes:")
        
        for query in test_queries:
            print(f"\n🔍 Test: '{query}'")
            
            # Mesurer la performance
            start_time = time.time()
            try:
                results = indexer.search(query, size=5)
                search_time = time.time() - start_time
                
                if results and 'hits' in results:
                    total_results = results['hits']['total']['value']
                    hits = results['hits']['hits']
                    
                    # Analyser les scores
                    scores = [hit['_score'] for hit in hits]
                    max_score = max(scores) if scores else 0
                    min_score = min(scores) if scores else 0
                    
                    query_result = {
                        'query': query,
                        'success': True,
                        'total_results': total_results,
                        'returned_results': len(hits),
                        'search_time': search_time,
                        'max_score': max_score,
                        'min_score': min_score
                    }
                    
                    print(f"   ✅ {total_results} résultats en {search_time:.3f}s")
                    print(f"   📊 Scores: {max_score:.2f} - {min_score:.2f}")
                    
                    # Détecter les problèmes
                    if search_time > 1.0:
                        issue = f"Recherche lente pour '{query}': {search_time:.3f}s"
                        diagnosis['performance_issues'].append(issue)
                        print(f"   ⚠️ Performance: Lent")
                    
                    if total_results == 0:
                        issue = f"Aucun résultat pour '{query}' - possible problème d'indexation"
                        diagnosis['content_issues'].append(issue)
                        print(f"   ⚠️ Contenu: Aucun résultat")
                    
                    if max_score < 1.0 and total_results > 0:
                        issue = f"Scores faibles pour '{query}' (max: {max_score:.2f}) - possible problème de pertinence"
                        diagnosis['content_issues'].append(issue)
                        print(f"   ⚠️ Pertinence: Scores faibles")
                
                else:
                    query_result = {
                        'query': query,
                        'success': False,
                        'search_time': search_time,
                        'error': 'Réponse vide ou invalide'
                    }
                    
                    issue = f"Erreur recherche pour '{query}': réponse invalide"
                    diagnosis['content_issues'].append(issue)
                    print(f"   ❌ Erreur: Réponse invalide")
            
            except Exception as e:
                search_time = time.time() - start_time
                query_result = {
                    'query': query,
                    'success': False,
                    'search_time': search_time,
                    'error': str(e)
                }
                
                issue = f"Exception recherche pour '{query}': {e}"
                diagnosis['performance_issues'].append(issue)
                print(f"   ❌ Exception: {e}")
            
            diagnosis['search_results'].append(query_result)
        
        # Générer des recommandations
        print(f"\n💡 ANALYSE ET RECOMMANDATIONS:")
        print("=" * 50)
        
        if diagnosis['performance_issues']:
            print(f"⚡ Problèmes de performance ({len(diagnosis['performance_issues'])}):")
            for issue in diagnosis['performance_issues']:
                print(f"   • {issue}")
            
            diagnosis['recommendations'].extend([
                "Redémarrer OpenSearch: docker-compose restart",
                "Vérifier l'utilisation mémoire et CPU",
                "Forcer le rafraîchissement des index",
                "Optimiser les index avec _forcemerge"
            ])
        
        if diagnosis['content_issues']:
            print(f"\n📄 Problèmes de contenu ({len(diagnosis['content_issues'])}):")
            for issue in diagnosis['content_issues']:
                print(f"   • {issue}")
            
            diagnosis['recommendations'].extend([
                "Vérifier que les documents sont correctement indexés",
                "Réindexer les documents avec les notebooks 02",
                "Vérifier les mappings des index",
                "Analyser la qualité du contenu indexé"
            ])
        
        if not diagnosis['performance_issues'] and not diagnosis['content_issues']:
            print(f"✅ Aucun problème détecté - Système de recherche opérationnel")
            diagnosis['recommendations'].append("Système fonctionnel - Monitoring régulier recommandé")
        
        if diagnosis['recommendations']:
            print(f"\n🔧 Recommandations:")
            for i, rec in enumerate(diagnosis['recommendations'], 1):
                print(f"   {i}. {rec}")
    
    except Exception as e:
        diagnosis['error'] = f"Erreur générale diagnostic: {e}"
        print(f"❌ Erreur diagnostic: {e}")
    
    return diagnosis

def check_common_issues() -> Dict:
    """Vérifier les problèmes courants"""
    
    print("🔍 Vérification des Problèmes Courants")
    print("=" * 60)
    
    issues_check = {
        'timestamp': datetime.now().isoformat(),
        'checks': [],
        'issues_found': [],
        'critical_issues': 0,
        'warnings': 0
    }
    
    # 1. Connexion OpenSearch
    print("\n1️⃣ Connexion OpenSearch:")
    try:
        response = requests.get(f"{OPENSEARCH_URL}/_cluster/health", timeout=5)
        if response.status_code == 200:
            health = response.json()
            status = health.get('status', 'unknown')
            
            check_result = {
                'check': 'opensearch_connection',
                'status': 'ok',
                'details': f"Cluster status: {status}"
            }
            
            print(f"   ✅ Connexion OK - Status: {status}")
            
            if status == 'red':
                issues_check['issues_found'].append("Cluster en status RED - Problème critique")
                issues_check['critical_issues'] += 1
                check_result['status'] = 'critical'
            elif status == 'yellow':
                issues_check['issues_found'].append("Cluster en status YELLOW - Attention requise")
                issues_check['warnings'] += 1
                check_result['status'] = 'warning'
        else:
            check_result = {
                'check': 'opensearch_connection',
                'status': 'error',
                'details': f"HTTP {response.status_code}"
            }
            issues_check['issues_found'].append(f"Connexion OpenSearch impossible: HTTP {response.status_code}")
            issues_check['critical_issues'] += 1
            print(f"   ❌ Erreur connexion: HTTP {response.status_code}")
    except Exception as e:
        check_result = {
            'check': 'opensearch_connection',
            'status': 'error',
            'details': str(e)
        }
        issues_check['issues_found'].append(f"Connexion OpenSearch impossible: {e}")
        issues_check['critical_issues'] += 1
        print(f"   ❌ Exception: {e}")
    
    issues_check['checks'].append(check_result)
    
    # 2. Index existants
    print("\n2️⃣ Index AskMe:")
    try:
        indexer = SimpleIndexer()
        all_indexes = indexer.list_indexes()
        askme_indexes = [idx for idx in all_indexes if idx.startswith('askme-') or idx == 'askme-documents']
        
        check_result = {
            'check': 'askme_indexes',
            'status': 'ok',
            'details': f"{len(askme_indexes)} index trouvés"
        }
        
        if askme_indexes:
            print(f"   ✅ {len(askme_indexes)} index AskMe trouvés")
            
            # Vérifier si des index sont vides
            empty_count = 0
            for idx in askme_indexes:
                idx_indexer = SimpleIndexer(index_name=idx)
                stats = idx_indexer.get_index_stats()
                if stats and stats.get('documents_count', 0) == 0:
                    empty_count += 1
            
            if empty_count > 0:
                issues_check['issues_found'].append(f"{empty_count} index vides détectés")
                issues_check['warnings'] += 1
                print(f"   ⚠️ {empty_count} index vides")
        else:
            issues_check['issues_found'].append("Aucun index AskMe trouvé")
            issues_check['critical_issues'] += 1
            check_result['status'] = 'critical'
            print(f"   ❌ Aucun index AskMe")
    except Exception as e:
        check_result = {
            'check': 'askme_indexes',
            'status': 'error',
            'details': str(e)
        }
        issues_check['issues_found'].append(f"Erreur vérification index: {e}")
        issues_check['critical_issues'] += 1
        print(f"   ❌ Erreur: {e}")
    
    issues_check['checks'].append(check_result)
    
    # 3. Test de recherche basique
    print("\n3️⃣ Test de recherche:")
    try:
        start_time = time.time()
        test_result = indexer.search("test", size=1)
        search_time = time.time() - start_time
        
        check_result = {
            'check': 'basic_search',
            'status': 'ok',
            'details': f"Recherche en {search_time:.3f}s"
        }
        
        print(f"   ✅ Recherche fonctionnelle ({search_time:.3f}s)")
        
        if search_time > 2.0:
            issues_check['issues_found'].append(f"Recherche très lente: {search_time:.3f}s")
            issues_check['warnings'] += 1
            check_result['status'] = 'warning'
            print(f"   ⚠️ Recherche lente")
    except Exception as e:
        check_result = {
            'check': 'basic_search',
            'status': 'error',
            'details': str(e)
        }
        issues_check['issues_found'].append(f"Erreur recherche: {e}")
        issues_check['critical_issues'] += 1
        print(f"   ❌ Erreur: {e}")
    
    issues_check['checks'].append(check_result)
    
    # Résumé
    print(f"\n📊 RÉSUMÉ:")
    print("=" * 40)
    print(f"🔍 Vérifications: {len(issues_check['checks'])}")
    print(f"❌ Problèmes critiques: {issues_check['critical_issues']}")
    print(f"⚠️ Avertissements: {issues_check['warnings']}")
    
    if issues_check['issues_found']:
        print(f"\n🚨 Problèmes détectés:")
        for issue in issues_check['issues_found']:
            print(f"   • {issue}")
    else:
        print(f"\n✅ Aucun problème détecté - Système opérationnel")
    
    return issues_check

# Exécuter les diagnostics
print("🔍 Diagnostic Complet du Système:")
print("=" * 60)

# Vérification des problèmes courants
common_issues = check_common_issues()

print(f"\n" + "="*60)

# Diagnostic avancé de recherche
search_diagnosis = diagnose_search_issues(["test", "configuration", "document"])

print(f"\n💡 Pour un diagnostic plus approfondi, consultez:")
print(f"   - Les logs Docker: docker-compose logs opensearch")
print(f"   - L'utilisation système: docker stats")
print(f"   - L'espace disque: df -h")

## 📋 Résumé - Maintenance et Monitoring

✅ **Fonctionnalités disponibles dans ce notebook :**

### 🏥 **Contrôle de Santé**
- `comprehensive_health_check()` - Contrôle complet du système
- `quick_system_check()` - Vérification rapide
- Monitoring du cluster, nœuds, index et performances
- Système d'alertes automatiques avec seuils configurables

### 📊 **Monitoring Continu**
- `monitor_system_metrics(duration, interval)` - Surveillance en temps réel
- Collecte des métriques: heap JVM, CPU, temps de recherche
- Analyse des tendances et détection d'anomalies
- Historique des mesures avec statistiques

### 🧹 **Maintenance**
- `analyze_storage_usage()` - Analyse détaillée de l'espace disque
- `cleanup_empty_indices()` - Nettoyage sécurisé des index vides
- `force_refresh_indices()` - Rafraîchissement forcé
- Recommandations d'optimisation automatiques

### 💾 **Sauvegarde et Récupération**
- `export_index_metadata()` - Export des métadonnées critiques
- `create_backup_script()` - Script de sauvegarde automatisé
- `disaster_recovery_plan()` - Plan de récupération d'urgence
- Stratégies de sauvegarde complètes

### 🔍 **Diagnostic et Dépannage**
- `diagnose_search_issues()` - Diagnostic approfondi des problèmes de recherche
- `check_common_issues()` - Vérification des problèmes fréquents
- Analyse des performances et de la pertinence
- Recommandations de résolution automatiques

### ⚠️ **Système d'Alertes**
- **Critiques**: Cluster rouge, connexion impossible, erreurs de recherche
- **Avertissements**: Cluster jaune, performances dégradées, heap élevé (>85%)
- **Informatifs**: Index vides, optimisations recommandées

### 📈 **Métriques Surveillées**
- **Cluster**: Status, nombre de nœuds, shards actifs/non-assignés
- **Performance**: Temps de recherche, utilisation CPU, heap JVM
- **Stockage**: Taille des index, nombre de documents, croissance
- **Santé**: Disponibilité, erreurs, temps de réponse

### 🔧 **Actions de Maintenance**
1. **Quotidiennes**: Vérification rapide, monitoring des alertes
2. **Hebdomadaires**: Contrôle complet, analyse du stockage
3. **Mensuelles**: Nettoyage, optimisation, sauvegarde complète
4. **Urgences**: Plan de récupération, diagnostic approfondi

### 💡 **Bonnes Pratiques**
- **Monitoring proactif**: Surveillance continue avec alertes
- **Sauvegardes régulières**: Métadonnées et procédures de restauration
- **Documentation**: Plans de récupération et historique des incidents
- **Tests réguliers**: Validation des performances et de la disponibilité

### 🚨 **Procédures d'Urgence**
1. **Cluster rouge**: Redémarrage, vérification logs, libération d'espace
2. **Performances dégradées**: Refresh, forcemerge, redémarrage
3. **Données perdues**: Restauration depuis sauvegarde, réindexation
4. **Corruption d'index**: Suppression/recréation, réindexation complète

🚀 **Le système de maintenance et monitoring est entièrement opérationnel !**

**💡 Utilisation recommandée :**
1. **Quotidien**: `quick_system_check()` dans vos scripts de monitoring
2. **Hebdomadaire**: `comprehensive_health_check()` pour analyse approfondie
3. **Mensuel**: `analyze_storage_usage()` et nettoyage si nécessaire
4. **Urgence**: `diagnose_search_issues()` et plan de récupération

**🔗 Intégration suggérée :**
- Scripts cron pour monitoring automatique
- Alertes par email/Slack sur seuils critiques
- Dashboard de monitoring avec métriques clés
- Documentation des procédures d'intervention